# Hyper Parameters Tuning

In [1]:
# Step 1: import required libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV,RandomizedSearchCV
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
# Step 2: Load the dataset
df = pd.read_csv('nvda_stock.csv')

In [3]:
# Step 3: Preprocess the data
df.head()

,Date,Open,High,Low,Close,Volume
0,2010-01-04,0.424242,0.426763,0.415074,0.423784,800204000
1,2010-01-05,0.422179,0.434556,0.422179,0.429972,728648000
2,2010-01-06,0.429743,0.433639,0.425617,0.432722,649168000
3,2010-01-07,0.430430,0.432264,0.421033,0.424242,547792000
4,2010-01-08,0.420804,0.428139,0.418283,0.425159,478168000


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4108 entries, 0 to 4107
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Date    4108 non-null   str    
 1   Open    4108 non-null   float64
 2   High    4108 non-null   float64
 3   Low     4108 non-null   float64
 4   Close   4108 non-null   float64
 5   Volume  4108 non-null   int64  
dtypes: float64(4), int64(1), str(1)
memory usage: 192.7 KB


In [5]:
# Drop the randoms rows 3108 rows from the dataset
df = df.drop(df.sample(n=3108, random_state=42).index)

In [6]:
df.shape

(1000, 6)

In [7]:
# handled the date column 

df['Date'] = pd.to_datetime(df['Date'])

df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day

# drop the date column after extracting year, month, and day
df = df.drop('Date', axis=1)

In [8]:
# divide the dataset into features and target variable
X = df.drop('Close', axis=1)
y = df['Close']


In [9]:
# split the dataset into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [10]:
# scale the features using StandardScaler

# create an instance of StandardScaler
scaler = StandardScaler()

#fit the scaler on the training data and transform both the training and testing data
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [11]:
# import model 

# create an instance of RandomForestRegressor

rf_model = RandomForestRegressor(random_state=42)

# train the model on the training data
rf_model.fit(X_train, y_train)

# predict the target variable on the testing data
rf_predictions = rf_model.predict(X_test)

# check the MSE and R2 score of the model

mse = mean_squared_error(y_test, rf_predictions)
r2 = r2_score(y_test, rf_predictions)

# print the MSE and R2 score
print("Mean Squared Error:", mse)
print("R2 Score:", r2)

Mean Squared Error: 0.8766733691989006
R2 Score: 0.9996852257305704


In [12]:
# check the training R2 score of the model

r2_train = r2_score(y_train, rf_model.predict(X_train))

# print the training R2 score
print("Training R2 Score:", r2_train)

Training R2 Score: 0.9999442400004328


In [13]:
# useing GridSearchCV for hyperparameter tuning

# make a dictionary of hyperparameters to tune

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

# make the grid search object

grid_search = GridSearchCV(estimator=rf_model, param_grid=param_grid, cv=3, n_jobs=-1, verbose=2)

# fit the grid search object to the training data
grid_search.fit(X_train, y_train)

# print the best hyperparameters
print("Best Hyperparameters:", grid_search.best_params_)
print("Best Score:", grid_search.best_score_)
    

Fitting 3 folds for each of 216 candidates, totalling 648 fits
Best Hyperparameters: {'bootstrap': True, 'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}
Best Score: 0.9994468672843005


In [14]:
# Using randomized search for hyperparameter tuning



# define the parameter distributions for randomized search
param_distributions = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

# make the randomized search object
randomized_search = RandomizedSearchCV(estimator=rf_model, param_distributions=param_distributions, n_iter=10, cv=3, n_jobs=-1, verbose=2)

# fit the randomized search object to the training data
randomized_search.fit(X_train, y_train)

# print the best hyperparameters
print("Best Hyperparameters:", randomized_search.best_params_)
print("Best Score:", randomized_search.best_score_)

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best Hyperparameters: {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': None, 'bootstrap': False}
Best Score: 0.9994129493695425
